# define

## TODO:
- salikt flagus, lai redzami vārdi kuriem vēl netika atrasts atbilstošs (vēlāk noderēs salīdzinot atvasinājumus)
- parādīt purpura krāsā ja pārbīdīts vārds, un par cik
- sāk meklēšanu kā tagad tieši no 0 ofseta, tad nākamo skatās +- pēdējais lietotias ofsets. ja nav, tad bīdās no o pa biškum un no ofseta pa biškum.


In [110]:
from pathlib import Path
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import difflib
import os
from typing import List, Dict, Tuple
import json
from datetime import datetime
import re
from lxml import etree
import json

import pandas as pd

nt_stubchapters = ["matthew", "mark", "luke", "john", "acts", "romans", "1_corinthians", "2_corinthians", "galatians", "ephesians", "philippians", "colossians", "1_thessalonians", "2_thessalonians", "1_timothy", "2_timothy", "titus", "philemon", "hebrews", "james", "1_peter", "2_peter", "1_john", "2_john", "3_john", "jude", "revelation"]

def element_to_dict(element):
    """Convert an lxml Element to a dictionary that can be JSON serialized"""
    result = {}
    
    if element.attrib:
        result['@attributes'] = dict(element.attrib)
    
    if element.text and element.text.strip():
        result['#text'] = element.text.strip()
    
    children = {}
    for child in element:
        if len(child) == 0:
            if child.text and child.text.strip():
                children[child.tag] = child.text.strip()
        else:
            children[child.tag] = element_to_dict(child)
    
    if children:
        result.update(children)
    
    return result

def extract_text_and_number(text):
    """
    Extract text and number from a string containing letters/spaces followed by a number.
    
    Args:
        text (str): Input string containing text and a number
        
    Returns:
        tuple: (text_part, number) or None if no valid pattern found
    """
    pattern = r'^([a-zA-Z\s\d]+)\s+(\d+)$'
    match = re.match(pattern, text.strip())
    return (match.group(1).strip().lower(), int(match.group(2))) if match else None

#def proiel_to_biblehub_chaps(text):
#    current_book, chapter = extract_text_and_number(text)
#    #return f"{chapter}_{current_book.lower()}" if current_book is not None else None

def tt(ch: str):
    return tr_ch_el_bh(ch)
def tr_ch_el_bh(ch: str):
    return ch.replace(" ", "_").lower()

class BibleSourceParser:
    def __init__(self):
        self.xml_parser = etree.XMLParser(resolve_entities=False, no_network=True)
        
    def parse_xml_source_proiel(self, xml_path: str, save_result= False) -> Dict[str, List[Dict]]:
        """Parse XML source from PROIEL project"""
        parser = self.xml_parser
        tree = etree.parse(xml_path, parser=parser)
        root = tree.getroot()
        
        result = {}
        bad_result = {}
        empty_result = {}
        df_result = []
        current_book = None
        
        # Use lxml's faster XPath expressions
        divs = root.xpath(".//div")
        ####attrdict = {}
        
        chdone = 0
        totchs = len(divs)
        for d in divs:
            current_book, chapter = extract_text_and_number(d.find("title").text.lower())
            result[f"{current_book}_{chapter}"] = []
            print(f"{current_book} {chapter} ( {float(chdone)/float(totchs)*100:.1f}% )")
            #print(current_book == "hebrews")
            
            verse_idx = 1
            sentences = d.xpath('sentence')
            for stn in range(len(sentences)):
                sentence = sentences[stn]
                #sentence_tokens = []
                word_idx = 0
                
                tokens = sentence.xpath('token')
                for ct in range(len(tokens)):
                    token = tokens[ct]
                    cittpartemtytokenswtf = token.attrib.get('citation-part', '')
                    
                    if not cittpartemtytokenswtf:
                        if f"{current_book}_{chapter}" not in bad_result:
                            bad_result[f"{current_book}_{chapter}"] = []
                        bad_result[f"{current_book}_{chapter}"].append(element_to_dict(token))
                        continue
                    emptytokenmarker = token.attrib.get('empty-token-sort', '')
                    if emptytokenmarker:
                        if f"{current_book}_{chapter}" not in empty_result:
                            empty_result[f"{current_book}_{chapter}"] = []
                        empty_result[f"{current_book}_{chapter}"].append(element_to_dict(token))
                        continue
                    #####for attr in token.attrib:
                        ####attrdict[attr] = f"token.attrib.get('{attr}', ''),"
                    versenum = int(token.attrib['citation-part'].split(".")[-1])
                    if(len(result[f"{current_book}_{chapter}"])<versenum):
                        for jj in range(len(result[f"{current_book}_{chapter}"]),versenum):
                            result[f"{current_book}_{chapter}"].append([])
                            #                    if(current_book == "Hebrews"):
                            #                        print(f"{versenum} -> {verse_idx}")
                            #if(versenum != verse_idx):
                            #    result[f"{current_book}_{chapter}"].append(sentence_tokens)
                            #    sentence_tokens =[]
                            #    verse_idx+=1
                    xdata = {
                        ######## text is inside form attribute(!)                        'text': token.text.strip(),
                        'verse': versenum,
                        'word': word_idx,
                        'id' : token.attrib.get('id', ''),
                        'form' : token.attrib.get('form', ''),
                        'citation-part' : token.attrib.get('citation-part', ''),
                        'lemma' : token.attrib.get('lemma', ''),
                        'part-of-speech' : token.attrib.get('part-of-speech', ''),
                        'morphology' : token.attrib.get('morphology', ''),
                        'head-id' : token.attrib.get('head-id', ''),
                        'relation' : token.attrib.get('relation', ''),
                        'presentation-after' : token.attrib.get('presentation-after', ''),
                        'information-status' : token.attrib.get('information-status', ''),
                        'empty-token-sort' : token.attrib.get('empty-token-sort', ''),
                        'antecedent-id' : token.attrib.get('antecedent-id', ''),
                        'presentation-before' : token.attrib.get('presentation-before', ''),
                        'contrast-group' : token.attrib.get('contrast-group', ''),
                        'sentence': stn,
                        ####"attrs": dict(token.attrib)
                    }
                    #sentence_tokens.append(xdata)
                    result[f"{current_book}_{chapter}"][versenum-1].append(xdata)
                    xdata['book'] = current_book
                    xdata['chapter'] = chapter
                    df_result.append(xdata)
                    word_idx += 1
                
            chdone += 1
            #result[f"{current_book}_{chapter}"].append(sentence_tokens)
        ####print(attrdict)
        if(save_result):
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"{timestamp}_badparses.txt"
            with open(filename, "w") as file:
                json.dump(bad_result, file, indent=2)
            filename = f"{timestamp}_emptytokens.txt"
            with open(filename, "w") as file:
                json.dump(empty_result, file, indent=2)
            dbname = f"{timestamp}_el_proiel_el.csv"
                # Create DataFrame and write to CSV
            df = pd.DataFrame(df_result)
            
            # Ensure output directory exists
            Path(dbname).parent.mkdir(parents=True, exist_ok=True)
            
            # Write to CSV with proper formatting
            df.to_csv(
                dbname,
                index=False,
                encoding='utf-8',
                quoting=1, 
                escapechar='\\'
            )
        return result

def str_getfrom_token_first(s1: str, tok: str):
    if not tok: return str
    pos = s1.find(tok)
    return s1[pos:] if pos >= 0 else s1

class BibleHTMLParser:
    def __init__(self):
        pass
    
    def parse_html_folder_biblehub(self, folder_path: str, book_filter: List[str]= None, save_result=False) -> Dict[str, List[List[str]]]:
        """Parse HTML files from BibleHub"""
        result = {}
        sentence_tokens = []
        df_result = []
        iterator_folders = []
        if book_filter is None:
            iterator_folders = os.listdir(folder_path)
        else:
            #if any(book.lower() in folder.lower() for book in book_filter)
            #dircont =
            iterator_folders = [n.lower() for n in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, n))
                       and n.lower() in book_filter]
            #for book in book_filter:
            #    if book.lower()  in dircont: iterator_folders.append(book)
        urls_greek = {}
        chdone = 0
        totchs = len(iterator_folders)
        for fold in iterator_folders:
            chapsnames = [filename for filename in os.listdir(folder_path.strip(os.path.sep)+os.path.sep+fold) 
                          if filename.endswith('.htm') or filename.endswith('.html')]
            ccchdone = 0
            totchhh = len(chapsnames)
            for filename in chapsnames:
                if(True):
                    chapter = os.path.splitext(filename)[0]
                    current_book = fold
                    result[f"{current_book}_{chapter}"] = []
                    print(f"{current_book} {chapter} ( {float(chdone)/float(totchs)*float(100)+(float(1)/float(totchs))*(float(ccchdone)/float(totchhh))*float(100):.1f}% )")
                    soup = BeautifulSoup(open(os.path.join(folder_path, fold, filename)), 'html.parser')
                    
                    table_spans = soup.find_all('table', class_='tablefloat')
                    curr_verse = 1
                    strong_num=-1
                    strong_title = ''
                    strong_en_title = ''
                    translit=''
                    translit_title=''
                    form = ''
                    form_en = ''
                    curr_node = None
                    word_idx = 0
                    for tbc in range(len(table_spans)):
                        table = table_spans[tbc]
                        #verse number?
                        curr_node = table.find("span", class_='reftop3')
                        if curr_node is not None:
                            # nbsp;
                            curr_verse = int(curr_node.text.strip("\xa0"))
                            word_idx = 0
                        
                        if(len(result[f"{current_book}_{chapter}"])<curr_verse):
                            for jj in range(len(result[f"{current_book}_{chapter}"]),curr_verse):
                                result[f"{current_book}_{chapter}"].append([])

                        #strongs title?
                        curr_node = table.find("span", class_='pos')
                        if curr_node is not None:
                            curr_node = curr_node.find('a')
                            if curr_node is not None:
                                strong_num = int(curr_node.text)
                                strong_title = curr_node.text + str_getfrom_token_first(curr_node.attrs.get('title', ''), ':')
                                attrb = curr_node.attrs.get('href', '')
                                if(attrb != ''):
                                    if(attrb.startswith('/greek')):
                                        urls_greek[attrb]=f"curl -O https://www.biblehub.com{attrb}"
                                        
                        #englishmans concordance
                        curr_node = table.find("span", class_='strongsnt2')
                        if curr_node is not None:
                            curr_node = curr_node.find('a')
                            if curr_node is not None:
                                strong_en_title = curr_node.text
                                attrb = curr_node.attrs.get('href', '')
                                if(attrb != ''):
                                    if(attrb.startswith('/greek')):
                                        urls_greek[attrb]=f"curl -O https://www.biblehub.com{attrb}"
                            
                        #translit
                        curr_node = table.find("span", class_='translit')
                        if curr_node is not None:
                            curr_node = curr_node.find('a')
                            if curr_node is not None:
                                attrb = curr_node.attrs.get('href', '')
                                if(attrb != ''):
                                    if(attrb.startswith('/greek')):
                                        urls_greek[attrb]=f"curl -O https://www.biblehub.com{attrb}"
                                translit_title = curr_node.attrs.get('title', '').replace("\xa0", ' ')
                                translit = curr_node.text.replace("\xa0", ' ')
                        
                        #greek text
                        curr_node = table.find("span", class_='greek')
                        if curr_node is not None:
                            form = curr_node.text.replace("\xa0", ' ')
                        
                        #en text
                        curr_node = table.find("span", class_='eng')
                        if curr_node is not None:
                            form_en = curr_node.text.replace("\xa0", ' ')
                                    
                        xdata = {
                        ######## text is inside form attribute(!)                        'text': token.text.strip(),
                        'verse': curr_verse,
                        'word': word_idx,
                        #'id' : token.attrib.get('id', ''),
                        'form' : form,
                        'form_en': form_en,
                        #'citation-part' : token.attrib.get('citation-part', ''),
                        #'lemma' : token.attrib.get('lemma', ''),
                        #'part-of-speech' : token.attrib.get('part-of-speech', ''),
                        #'morphology' : token.attrib.get('morphology', ''),
                        #'head-id' : token.attrib.get('head-id', ''),
                        #'relation' : token.attrib.get('relation', ''),
                        #'presentation-after' : token.attrib.get('presentation-after', ''),
                        #'information-status' : token.attrib.get('information-status', ''),
                        #'empty-token-sort' : token.attrib.get('empty-token-sort', ''),
                        #'antecedent-id' : token.attrib.get('antecedent-id', ''),
                        #'presentation-before' : token.attrib.get('presentation-before', ''),
                        #'contrast-group' : token.attrib.get('contrast-group', ''),
                        ####"attrs": dict(token.attrib)
                        'strong_num': strong_num,
                        'strong_title' : strong_title,
                        'strong_en_title' : strong_en_title,
                        'translit': translit,
                        'translit_title': translit_title
                    }
                        result[f"{current_book}_{chapter}"][curr_verse-1].append(xdata)
                        xdata['book'] = current_book
                        xdata['chapter'] = chapter
                        df_result.append(xdata)
                        word_idx += 1
                ccchdone +=1
            chdone += 1

        if(save_result):
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            filename = f"{timestamp}_dload.sh"
            with open(filename, "w") as file:
                for k, v in urls_greek.items():
                    file.write(f"{v}\n")
            dbname = f"{timestamp}_biblehub_el_en_direct.csv"
                # Create DataFrame and write to CSV
            df = pd.DataFrame(df_result)
            
            # Ensure output directory exists
            Path(dbname).parent.mkdir(parents=True, exist_ok=True)
            
            # Write to CSV with proper formatting
            df.to_csv(
                dbname,
                index=False,
                encoding='utf-8',
                quoting=1, 
                escapechar='\\'
            )                    
        return result

class BibleComparator:
    def __init__(self):
        self.parser_xml = BibleSourceParser()
        self.parser_html = BibleHTMLParser()
        
    def compare_verses(self, xml_source: Dict[str, List[List[Dict]]], 
                      html_source: Dict[str, List[List[str]]]) -> List[Tuple]:
        """Compare verses and generate comparison tuples"""
        comparisons = []
        
        for book_chapter in xml_source.keys():
            if book_chapter in html_source:
                xml_verses = xml_source[book_chapter]
                html_verses = html_source[book_chapter]
                
                max_verses = max(len(xml_verses), len(html_verses))
                
                for verse_num in range(max_verses):
                    xml_verse = xml_verses[verse_num] if verse_num < len(xml_verses) else None
                    html_verse = html_verses[verse_num] if verse_num < len(html_verses) else None
                    
                    comparisons.append((book_chapter, verse_num + 1, xml_verse, html_verse))
        
        return comparisons
    
    def generate_html_report(self, comparisons: List[Tuple], 
                           output_path: str = 'comparison_report.html'):
        """Generate HTML report with interlinear comparison"""
        html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Bible Greek Sources Comparison</title>
    <style>
        .container { margin: 20px; font-family: Arial, sans-serif; }
        .verse-comparison { border-collapse: collapse; width: 100%; margin-bottom: 20px; }
        .verse-comparison td { padding: 8px; border: 1px solid #ddd; }
        .source-header { background-color: #f5f5f5; }
        .difference { background-color: #ffebee; }
        .addition { background-color: #e8f5e9; }
        .deletion { background-color: #ffebee; }
        .nav-buttons { margin-top: 10px; }
        .nav-button { padding: 5px 10px; margin-right: 5px; }
    </style>
</head>
<body>
    <div class="container">
        {}
    </div>
</body>
</html>
"""
        
        report_content = []
        
        for book_chapter, verse_num, xml_verse, html_verse in comparisons:
            report_content.append(f'<h2>{book_chapter}, Verse {verse_num}</h2>')
            
            if xml_verse and html_verse:
                # Compare verses
                diff = difflib.SequenceMatcher(None, 
                                             ' '.join(t['form'] for t in xml_verse),
                                             ' '.join(html_verse))
                
                # Generate comparison table
                table_rows = ['<table class="verse-comparison">']
                table_rows.append('<tr><th>PROIEL Source</th><th>BibleHub Source</th></tr>')
                
                # Get aligned sequences
                opcodes = diff.get_opcodes()
                for opcode in opcodes:
                    if opcode[0] == 'equal':
                        xml_words = ' '.join(t['form'] for t in xml_verse[opcode[1]:opcode[2]])
                        html_words = ' '.join(html_verse[opcode[3]:opcode[4]])
                        table_rows.append(f'<tr><td>{xml_words}</td><td>{html_words}</td></tr>')
                    elif opcode[0] == 'replace':
                        xml_words = ' '.join(t['form'] for t in xml_verse[opcode[1]:opcode[2]])
                        html_words = ' '.join(html_verse[opcode[3]:opcode[4]])
                        table_rows.append(f'<tr><td class="difference">{xml_words}</td><td class="difference">{html_words}</td></tr>')
                    elif opcode[0] == 'delete':
                        xml_words = ' '.join(t['form'] for t in xml_verse[opcode[1]:opcode[2]])
                        table_rows.append(f'<tr><td class="deletion">{xml_words}</td><td></td></tr>')
                    elif opcode[0] == 'insert':
                        html_words = ' '.join(html_verse[opcode[3]:opcode[4]])
                        table_rows.append(f'<tr><td></td><td class="addition">{html_words}</td></tr>')
                
                table_rows.append('</table>')
                report_content.append('\n'.join(table_rows))
            
            # Add navigation buttons
            report_content.append("""
                <div class="nav-buttons">
                    <button onclick="window.location.href='#'" class="nav-button">Top</button>
                    <button onclick="history.back()" class="nav-button">Previous</button>
                    <button onclick="history.forward()" class="nav-button">Next</button>
                </div>
            """)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(html_template.format('\n'.join(report_content)))


In [ ]:
#!mamba install -y lxml
#!mamba install -y pandas

# start work

In [111]:
comparator = BibleComparator()
#html_sources = comparator.parser_html.parse_html_folder_biblehub('../../00rebordn2/', nt_stubchapters, True)
html_sources = comparator.parser_html.parse_html_folder_biblehub('../../00rebordn2/', ["hebrews"], False)


hebrews 9 ( 0.0% )
hebrews 5 ( 7.7% )
hebrews 8 ( 15.4% )
hebrews 3 ( 23.1% )
hebrews 4 ( 30.8% )
hebrews 13 ( 38.5% )
hebrews 11 ( 46.2% )
hebrews 7 ( 53.8% )
hebrews 1 ( 61.5% )
hebrews 2 ( 69.2% )
hebrews 12 ( 76.9% )
hebrews 6 ( 84.6% )
hebrews 10 ( 92.3% )


In [ ]:
#print(html_sources.keys())
#print(([j['verse'] for i in html_sources['mark_12'] for j in i if j.get('verse') == '1']))
#talist = [
#    itm 
#          for j in html_sources['mark_12'] 
#          for itm in j 
#          if int(itm['verse']) < 2
#          ]

#print(len(talist))
#print(talist[12])
print(html_sources['mark_12'][0][12])
print(int(html_sources['mark_12'][0][12]['verse']) <2)
#for i in range(1):
#    for j in range(30):
##        print(html_sources['mark_12'][i][j].get('verse')=='1')
#        print(len(html_sources['mark_12'][i][j]))

In [112]:
#proiel_to_biblehub_chaps(nt_stubchapters))
xml_sources = comparator.parser_xml.parse_xml_source_proiel('./greek-nt.xml', True)#True)
#xml_sources = comparator.parser_xml.parse_xml_source_proiel('./greek-nt-syn.xml', True)#True)


matthew 1 ( 0.0% )
matthew 2 ( 0.4% )
matthew 3 ( 0.8% )
matthew 4 ( 1.2% )
matthew 5 ( 1.6% )
matthew 6 ( 2.0% )
matthew 7 ( 2.4% )
matthew 8 ( 2.8% )
matthew 9 ( 3.2% )
matthew 10 ( 3.6% )
matthew 11 ( 4.0% )
matthew 12 ( 4.4% )
matthew 13 ( 4.8% )
matthew 14 ( 5.2% )
matthew 15 ( 5.6% )
matthew 16 ( 6.0% )
matthew 17 ( 6.5% )
matthew 18 ( 6.9% )
matthew 19 ( 7.3% )
matthew 20 ( 7.7% )
matthew 21 ( 8.1% )
matthew 22 ( 8.5% )
matthew 23 ( 8.9% )
matthew 24 ( 9.3% )
matthew 25 ( 9.7% )
matthew 26 ( 10.1% )
matthew 27 ( 10.5% )
matthew 28 ( 10.9% )
mark 1 ( 11.3% )
mark 2 ( 11.7% )
mark 3 ( 12.1% )
mark 4 ( 12.5% )
mark 5 ( 12.9% )
mark 6 ( 13.3% )
mark 7 ( 13.7% )
mark 8 ( 14.1% )
mark 9 ( 14.5% )
mark 10 ( 14.9% )
mark 11 ( 15.3% )
mark 12 ( 15.7% )
mark 13 ( 16.1% )
mark 14 ( 16.5% )
mark 15 ( 16.9% )
mark 16 ( 17.3% )
luke 1 ( 17.7% )
luke 2 ( 18.1% )
luke 3 ( 18.5% )
luke 4 ( 19.0% )
luke 5 ( 19.4% )
luke 6 ( 19.8% )
luke 7 ( 20.2% )
luke 8 ( 20.6% )
luke 9 ( 21.0% )
luke 10 ( 21.4

# check data compatibility

In [109]:
def tt(ch: str):
    return tr_ch_el_bh(ch)
def tr_ch_el_bh(ch: str):
    return ch.replace(" ", "_").lower()

def ta(ch: str):
    return tr_ch_bh_el(ch)
def tr_ch_bh_el(ch: str):
    #return ch.replace("_", " ").lower()
    return ch[0].upper()+ ch[1:]

print(xml_sources.keys())
print(html_sources.keys())
print(ta('mark_12'))
print(xml_sources[ta('mark_12')][2][0])
print(html_sources[('mark_12')][0][2])
#print(
#[item for list in xml_sources[ta('mark_12')] for item in  list
#          if item.get('verse') == '2' and item.get('word') == '0']

#)

dict_keys(['Matthew_1', 'Matthew_2', 'Matthew_3', 'Matthew_4', 'Matthew_5', 'Matthew_6', 'Matthew_7', 'Matthew_8', 'Matthew_9', 'Matthew_10', 'Matthew_11', 'Matthew_12', 'Matthew_13', 'Matthew_14', 'Matthew_15', 'Matthew_16', 'Matthew_17', 'Matthew_18', 'Matthew_19', 'Matthew_20', 'Matthew_21', 'Matthew_22', 'Matthew_23', 'Matthew_24', 'Matthew_25', 'Matthew_26', 'Matthew_27', 'Matthew_28', 'Mark_1', 'Mark_2', 'Mark_3', 'Mark_4', 'Mark_5', 'Mark_6', 'Mark_7', 'Mark_8', 'Mark_9', 'Mark_10', 'Mark_11', 'Mark_12', 'Mark_13', 'Mark_14', 'Mark_15', 'Mark_16', 'Luke_1', 'Luke_2', 'Luke_3', 'Luke_4', 'Luke_5', 'Luke_6', 'Luke_7', 'Luke_8', 'Luke_9', 'Luke_10', 'Luke_11', 'Luke_12', 'Luke_13', 'Luke_14', 'Luke_15', 'Luke_16', 'Luke_17', 'Luke_18', 'Luke_19', 'Luke_20', 'Luke_21', 'Luke_22', 'Luke_23', 'Luke_24', 'John_1', 'John_2', 'John_3', 'John_4', 'John_5', 'John_6', 'John_7', 'John_8', 'John_9', 'John_10', 'John_11', 'John_12', 'John_13', 'John_14', 'John_15', 'John_16', 'John_17', 'John_

KeyError: 'mark_12'

In [ ]:
print(xml_sources[ta('hebrews_9')][26][1])
print(html_sources[('hebrews_9')][26][1])

In [ ]:
len(html_sources[('hebrews_9')][0])

In [114]:
print(len(xml_sources[('hebrews_9')]))
print(len(xml_sources[('hebrews_9')][20]))
print((xml_sources[('hebrews_9')][20][0]))

28
14
{'verse': 21, 'word': 0, 'id': '469734', 'form': 'καὶ', 'citation-part': 'HEB 9.21', 'lemma': 'καί', 'part-of-speech': 'C-', 'morphology': '---------n', 'head-id': '469747', 'relation': 'obj', 'presentation-after': ' ', 'information-status': '', 'empty-token-sort': '', 'antecedent-id': '', 'presentation-before': '', 'contrast-group': '', 'sentence': 14, 'book': 'hebrews', 'chapter': 9}


In [ ]:
for i in range(28):
    print((xml_sources[ta('hebrews_9')][i][0]))

In [ ]:
news = {}
ii = 0
for bk in html_sources.keys():
    for i in xml_sources[ta(bk)]:
        print(bk)
        print(len(i))
        verse_tokens = []
        for j in i:
            print(len(j))
            verse_tokens.append(html_sources[bk][0][ii])
            ii +=1
        news[bk].append(verse_tokens)


print(xml_sources[ta('mark_12')][2][0])
print(news[('mark_12')][2][0])

In [ ]:
print(xml_sources["Hebrews_9"][17])

In [ ]:
print(html_sources.keys())
print(news.keys())

In [ ]:
news = {}
ii = 0
for bk in html_sources.keys():
    for i in xml_sources[ta(bk)]:
        print(bk)
        print(len(i))
        verse_tokens = []
        for j in i:
            print(len(j))
            verse_tokens.append(html_sources[bk][0][ii])
            ii +=1
        news[bk].append(verse_tokens)


print(xml_sources[ta('mark_12')][2][0])
print(news[('mark_12')][2][0])

# check hebs

In [104]:
print(xml_sources[ta('hebrews_9')][26][1])
print(html_sources[('hebrews_9')][26][1])

samecnt = 0
totalcnt = 0
diffcnt = 0
diffids =[]
for iii in range(1, 14):
    for jjj in range (len(xml_sources[ta('hebrews_'+str(iii))])):
        for kkk in range (len(xml_sources[ta('hebrews_'+str(iii))][jjj])):
            totalcnt+=1
            x = xml_sources[ta('hebrews_'+str(iii))][jjj][kkk]
            print('hebrews_'+str(iii)+'::'+str(jjj)+'->'+str(kkk))
            y = html_sources['hebrews_'+str(iii)][jjj][kkk]
            if(x == y):
                samecnt+=1
            else:
                diffcnt +=1
                diffids.append(((iii, jjj, kkk), (x, y)))
print(f"total: {totalcnt} same: {samecnt} diff: {diffcnt}")
print(f"total: {totalcnt} same: {float(samecnt)/float(totalcnt)*float(100):.1f}% diff: {float(diffcnt)/float(totalcnt)*float(100):.1f}%")
    

{'verse': 27, 'word': 1, 'id': '469859', 'form': 'καθ’', 'citation-part': 'HEB 9.27', 'lemma': 'κατά', 'part-of-speech': 'R-', 'morphology': '---------n', 'head-id': '469861', 'relation': 'adv', 'presentation-after': ' ', 'information-status': '', 'empty-token-sort': '', 'antecedent-id': '', 'presentation-before': '', 'contrast-group': '', 'sentence': 20, 'book': 'Hebrews', 'chapter': 9}
{'verse': 27, 'word': 1, 'form': 'καθ’', 'form_en': 'in', 'strong_num': 2596, 'strong_title': '2596: A primary particle; down, in varied relations (genitive, dative or accusative) with which it is joined).', 'strong_en_title': '[e]', 'translit': 'kath’', 'translit_title': 'kath’: A primary particle; down, in varied relations (genitive, dative or accusative) with which it is joined).', 'book': 'hebrews', 'chapter': '9'}
hebrews_1::0->0
hebrews_1::0->1
hebrews_1::0->2
hebrews_1::0->3
hebrews_1::0->4
hebrews_1::0->5
hebrews_1::0->6
hebrews_1::0->7
hebrews_1::0->8
hebrews_1::0->9
hebrews_1::0->10
hebrews_1

IndexError: list index out of range

In [105]:
iii = 1
jjj = 0
print(f"ht: {len(html_sources['hebrews_'+str(iii)][jjj])}") #[kkk]))
print(f"xm: {len(xml_sources[ta('hebrews_'+str(iii))][jjj])}") #[kkk]))

ht: 12
xm: 21


# visul rende

In [126]:
#mport copy
xm_flags ={}
hm_flags = {k: [] for k in html_sources.keys()}

for k in xml_sources.keys():
    xm_flags[k] = []
    for c in range(len(xml_sources[k])):
        for v in range(len(xml_sources[k][c])):
            xm_flags[k].append([])
            for w in range(len(xml_sources[k][c][v])):
                xm_flags[k][v].append([])
                xm_flags[k][v][w] = False

for k in html_sources.keys():
    hm_flags[k] = []
    for c in range(len(html_sources[k])):
        for v in range(len(html_sources[k][c])):
            hm_flags[k].append([])
            for w in range(len(html_sources[k][c][v])):
                hm_flags[k][v].append([])
                hm_flags[k][v][w] = False                

print(xm_flags.keys())
print(hm_flags.keys())


dict_keys(['matthew_1', 'matthew_2', 'matthew_3', 'matthew_4', 'matthew_5', 'matthew_6', 'matthew_7', 'matthew_8', 'matthew_9', 'matthew_10', 'matthew_11', 'matthew_12', 'matthew_13', 'matthew_14', 'matthew_15', 'matthew_16', 'matthew_17', 'matthew_18', 'matthew_19', 'matthew_20', 'matthew_21', 'matthew_22', 'matthew_23', 'matthew_24', 'matthew_25', 'matthew_26', 'matthew_27', 'matthew_28', 'mark_1', 'mark_2', 'mark_3', 'mark_4', 'mark_5', 'mark_6', 'mark_7', 'mark_8', 'mark_9', 'mark_10', 'mark_11', 'mark_12', 'mark_13', 'mark_14', 'mark_15', 'mark_16', 'luke_1', 'luke_2', 'luke_3', 'luke_4', 'luke_5', 'luke_6', 'luke_7', 'luke_8', 'luke_9', 'luke_10', 'luke_11', 'luke_12', 'luke_13', 'luke_14', 'luke_15', 'luke_16', 'luke_17', 'luke_18', 'luke_19', 'luke_20', 'luke_21', 'luke_22', 'luke_23', 'luke_24', 'john_1', 'john_2', 'john_3', 'john_4', 'john_5', 'john_6', 'john_7', 'john_8', 'john_9', 'john_10', 'john_11', 'john_12', 'john_13', 'john_14', 'john_15', 'john_16', 'john_17', 'john_

In [125]:
xm_flags["hebrews_9"][1][1]

False

In [115]:
def ta(s):
    return s
#print(f"x: {xml_sources[ta("hebrews_1")][0][0]}")
#print(f"h: {html_sources["hebrews_1"][0][0]}")
# ANSI color codes
GREEN = '\033[92m'
RED = '\033[91m'
#ESC[38:5:⟨n⟩m Select foreground color      where n is a number from the table below
#ESC[48:5:⟨n⟩m Select background color
RED_W =  '\x1b[38;5;88m\x1b[48;5;7m'
DIFF_BLUE =  '\x1b[38;5;12m\x1b[48;15;7m'
DIFF_RED =  '\x1b[38;5;196m\x1b[48;15;7m'
#default bg, dark red letter
#'\x1b[38;5;88m'
#blue light bg, white letter
#'\x1b[5;46;88m'
#green bg whitel etter
#'\x1b[5;46;42m'
#green bg oragneleter'\x1b[5;31;42m'
# #'\033[38;5;55'
YELLOW = '\033[33m'
RESET = '\033[0m'
COLOR = RESET

class Visualizer_Cosole():
    def __init__(self):
        self.similars ={
    "θ" : "Θ",
    "Θ" : "θ"
}
        pass
    
    def letters_same(self, l1, l2):
        return self.words_same(l1, l2)
    
    def letters_similiar(self, l1, l2):
        if(l1.lower() == l2.lower()): return True
        sim = self.similars.get(l1.lower(), '').lower()
        if(sim!= ''): return sim == l2.lower()
        sim = self.similars.get(l2.lower(), '').lower()
        if(sim!= ''): return sim == l1.lower()
        return False
    
    def words_same(self, w1, w2):
        return w1 ==w2
    
    def getdiff_word_Visual(self, v, w, xx: str, yh: str, showGreens: bool = False, showYellows: bool = False)-> tuple[bool, bool, str]:
        p1 =""
        hadRed = False
        allReds = True
        COLOR = ''
        if(self.words_same(xx,yh)):
            allReds = False
            if not showGreens: return (hadRed, allReds, '')
            #COLOR = GREEN
            #s1 = f"{xx}"
            #s2 = f"{yh}{RESET}"
            return (hadRed, allReds, self.w_colordiff_OK(v, w, xx, yh))
        else:
            
            minlen = min(len(xx), len(yh))
            s1=""
            s2=""
            for i in range(minlen):
                if(xx[i]==yh[i]):
                    allReds = False
                    s1+=f"{GREEN}{xx[i]}{RESET}"
                    s2+=f"{GREEN}{yh[i]}{RESET}"
                    continue
                else:
                    if(self.letters_similiar(xx[i], yh[i])):
                        allReds = False
                        s1+=f"{YELLOW}{xx[i]}{RESET}"
                        s2+=f"{YELLOW}{yh[i]}{RESET}"
                        continue
                    hadRed = True
                    s1+=f"{RED}{xx[i]}{RESET}"
                    s2+=f"{RED}{yh[i]}{RESET}"
            #print(f"minlen: {minlen}")
            if(len(xx)> minlen):
                s1+=f"{RED}{xx[minlen:]}{RESET}"
            if(len(yh)> minlen):
                s2+=f"{RED}{yh[minlen:]}{RESET}"
            if(True):
                if(hadRed):
                    COLOR = RED
                else:
                    if not showYellows: return (hadRed, allReds, '')
                    COLOR = YELLOW
        p1 = f"{COLOR}{v}:{w}{RESET}"
        return (hadRed, allReds, f"{p1}\n{s1}\n{s2}")
    
    def w_colordiff_OK(self, v, w, xx: str, yh: str)-> str:
        COLOR = GREEN
        s1 = f"{xx}"
        s2 = f"{yh}{RESET}"
        p1 = f"{COLOR}{v}:{w}{RESET}"
        return f"{p1}\n{s1}\n{s2}"
            


seperator= ' '
dw = Visualizer_Cosole()
book = "hebrews"
ch_nr = 1
chapt = book+"_"+str(ch_nr)
#v = 0
xm_c_len = len(xml_sources[ta(chapt)])
hm_c_len = len(html_sources[chapt])

minverses = min(xm_c_len, hm_c_len)
for v in range(minverses):
    w = 0
    xm_v_len = len(xml_sources[ta(chapt)][v])
    hm_v_len = len(html_sources[chapt][v])
    print(f"~~~~~~b:___ {book[:3].upper()} {ch_nr}:{v+1} ___:d~~~~~~")
    shorter_x = False
    if(xm_v_len!= hm_v_len):
        if(xm_v_len<hm_v_len): shorter_x = True
        print(f"{RED_W}xml: {xm_v_len} w\nhml: {hm_v_len} w ====== ({float(xm_v_len-hm_v_len)/float(xm_v_len+hm_v_len)*float(100):.1f}% )\n{RESET}~~~~~")
    else:
        print(f"{GREEN}wrd: {xm_v_len}\n{RESET}~~~~~")
    minwords = min (xm_v_len, hm_v_len)
    for i in range (minwords):
        w = i
        xx = xml_sources[ta(chapt)][v][w]["form"]
        yh = html_sources[chapt][v][w]["form"]
        hR, aR, resultGUI = dw.getdiff_word_Visual(v+1, w+1, xx, yh, showYellows=True)
        if(resultGUI):
            print(resultGUI)
    if(shorter_x):
        strbuild =''
        for i in range(xm_v_len, hm_v_len):
            strbuild += f"{html_sources[chapt][v][i]['form']}{seperator}"
        print(f"{DIFF_RED}+{strbuild}{RESET}")
    else : 
        if (hm_v_len < xm_v_len):
            strbuild =''
            
            for i in range(hm_v_len, xm_v_len):
                strbuild += f"{xml_sources[ta(chapt)][v][i]['form']}{seperator}"
            print(f"{DIFF_BLUE}-{strbuild}{RESET}")
# write a function to visualize the differences in the console                      
            
        


~~~~~~b:___ HEB 1:1 ___:d~~~~~~
xml: 21 w
hml: 12 w ====== (27.3% )
~~~~~
1:6
θεὸς
Θεὸς
-ἐπ’ ἐσχάτου τῶν ἡμερῶν τούτων ἐλάλησεν ἡμῖν ἐν υἱῷ 
~~~~~~b:___ HEB 1:2 ___:d~~~~~~
xml: 10 w
hml: 19 w ====== (-31.0% )
~~~~~
2:1
ὃν
ἐπ’
2:2
ἔθηκεν
ἐσχάτου
2:3
κληρονόμον
τῶν
2:4
πάντων
ἡμερῶν
2:5
δι’
τούτων
2:6
οὗ
ἐλάλησεν
2:7
καὶ
ἡμῖν
2:8
ἐποίησεν
ἐν
2:9
τοὺς
Υἱῷ
2:10
αἰῶνας
ὃν
+ἔθηκεν κληρονόμον πάντων δι’ οὗ καὶ ἐποίησεν τοὺς αἰῶνας 
~~~~~~b:___ HEB 1:3 ___:d~~~~~~
xml: 30 w
hml: 31 w ====== (-1.6% )
~~~~~
3:19
αὐτοῦ
‹δι’›
3:20
καθαρισμὸν
αὐτοῦ
3:21
τῶν
καθαρισμὸν
3:22
ἁμαρτιῶν
τῶν
3:23
ποιησάμενος
ἁμαρτιῶν
3:24
ἐκάθισεν
ποιησάμενος
3:25
ἐν
ἐκάθισεν
3:26
δεξιᾷ
ἐν
3:27
τῆς
δεξιᾷ
3:28
μεγαλωσύνης
τῆς
3:29
ἐν
Μεγαλωσύνης
3:30
ὑψηλοῖς
ἐν
+ὑψηλοῖς 
~~~~~~b:___ HEB 1:4 ___:d~~~~~~
wrd: 11
~~~~~
~~~~~~b:___ HEB 1:5 ___:d~~~~~~
wrd: 27
~~~~~
5:1
τίνι
Τίνι
5:7
υἱός
Υἱός
5:15
καὶ
Καὶ
5:17
ἐγὼ
Ἐγὼ
5:21
πατέρα
Πατέρα
5:27
υἱόν
Υἱόν
~~~~~~b:___ HEB 1:6 ___:d~~~~~~
wrd: 16
~~~~~
6:1
ὅταν
Ὅταν
6:11
καὶ
Καὶ
6

In [ ]:
print(getdiff_word_Visual(0, 0, "akkara", "", showGreens = False))
print("absdef"[0:])

In [ ]:
print(xml_sources[ta(chapt)][10][5]["form"])
print(html_sources[(chapt)][10][5]["form"])

```
v = 0; w=5
print(len(xml_sources[ta("hebrews_1")][v][w]["form"]))
print(len(html_sources["hebrews_1"][v][w]["form"]))

for i in range(4):
    print(f"{xml_sources[ta("hebrews_1")][v][w]["form"][i] == html_sources["hebrews_1"][v][w]["form"][i]}")
```

## all check

In [ ]:

comparisons = comparator.compare_verses(xml_sources, html_sources)
comparator.generate_html_report(comparisons)